# Lumen 1.3 SFT — Qwen3.5-9B QLoRA (Unsloth)

Trains Lumen on public data pulled via `scripts/fetch-lumen-data.mjs`: SWE-smith +
SWE-agent-plus (agentic SWE, reasoning-backfilled), Magicoder (code-instruct),
Tulu-3 (general), Hermes (tool-calling), and OpenCodeReasoning (native `<think>`
reasoning traces).

**Base model is multimodal** (text + image + video in, text out). We fine-tune the
language side only and keep the vision tower frozen — vision ability is inherited
from the base model for free, and training it on our 100%-text data could only
degrade it. Serving multimodal input (vLLM config, Worker `image_url` parts,
chat.html upload UI) is a separate workstream, not needed to train.

**Note:** Qwen3.5 has thinking mode ON by default (Qwen3 was opt-in), so the
`<think>` coverage in the mix matters more than it did — see the reasoning slice.

**Workflow (units are precious):**
1. Upload `models/Lumen/JSONLs (DATASETS)/In Use/lumen-*-1.3.jsonl` to Google Drive
   folder `lumen-data/` (or Kaggle dataset `lumen-data`).
2. Run this notebook END-TO-END with `TINY = True` on a **free GPU** (Kaggle T4 / Colab free T4) until it finishes without errors.
3. Only then: Colab **A100**, `TINY = False`. Expect a few hours for one epoch on ~55k samples.

Checkpoints go to Drive every `SAVE_STEPS` steps — a disconnect loses minutes, not the run.
Resume by re-running with `RESUME = True`.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
TINY        = True          # True = 200-sample dry run (debug on free GPU first!)
RESUME      = False         # resume from last checkpoint in OUTPUT_DIR
BASE_MODEL  = 'unsloth/Qwen3.5-9B'   # multimodal: text+image+video in, text out
MAX_SEQ     = 8192          # A100 40GB handles 8192 comfortably with QLoRA
LORA_R      = 32
LORA_ALPHA  = 64
LR          = 2e-4
EPOCHS      = 1
BATCH       = 2             # per-device
GRAD_ACCUM  = 16            # effective batch 32
SAVE_STEPS  = 200
SEED        = 3407

# Qwen3.5 is a VLM. Our data is 100% text, so the vision tower stays FROZEN —
# training it against zero visual signal would only degrade inherited vision.
# Vision ability comes free from the base model; we only tune the language side.
FINETUNE_VISION = False

# Mix ratios by slice (task field -> target share). Slices that run out are
# used fully; shares renormalize over what exists.
#
# Reasoning is weighted heavily (0.25) because Qwen3.5 has thinking ON by default:
# too much non-thinking data would actively suppress the behavior we want to keep.
# The swe-agentic slice is *also* reasoning-bearing wherever the -reasoning
# backfilled variants are present, so true <think> coverage is higher than 0.25.
MIX = {
    'swe-agentic':  0.35,   # lumen-swe-smith-1.3 / lumen-swe-agent-plus-1.3 (+ -reasoning variants)
    'reasoning':    0.25,   # lumen-open-code-reasoning-1.3 (native <think> traces)
    'general':      0.17,   # lumen-tulu-1.3
    'code-instruct':0.12,   # lumen-magicoder-1.3
    'tool-calling': 0.11,   # lumen-hermes-1.3
}
TARGET_TOTAL = 200 if TINY else 55_000

In [ ]:
# ── Install ────────────────────────────────────────────────────────────────
%pip install -q unsloth
import torch
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9), 'GB')

In [ ]:
# ── Locate data (Colab Drive or Kaggle input) ──────────────────────────────
import os, glob
if os.path.exists('/kaggle'):
    DATA_DIR   = '/kaggle/input/lumen-data'          # add the dataset to the notebook
    OUTPUT_DIR = '/kaggle/working/lumen-ckpt'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR   = '/content/drive/MyDrive/lumen-data'
    OUTPUT_DIR = '/content/drive/MyDrive/lumen-ckpt'  # checkpoints survive disconnects
files = sorted(glob.glob(f'{DATA_DIR}/*.jsonl'))
assert files, f'No .jsonl files found in {DATA_DIR} — upload models/Lumen/JSONLs (DATASETS)/In Use/*.jsonl there.'
print('\n'.join(files))

In [ ]:
# ── Load slices, build the mix ─────────────────────────────────────────────
import json, random
random.seed(SEED)

SLICE_OF_FILE = {  # filename stem -> mix key
    'lumen-swe-smith-1.3':            'swe-agentic',
    'lumen-swe-smith-reasoning-1.3':  'swe-agentic',
    'lumen-swe-agent-plus-1.3':       'swe-agentic',
    'lumen-swe-agent-plus-reasoning-1.3': 'swe-agentic',
    'lumen-magicoder-1.3':            'code-instruct',
    'lumen-tulu-1.3':                 'general',
    'lumen-hermes-1.3':               'tool-calling',
    'lumen-open-code-reasoning-1.3':  'reasoning',
}

# If a reasoning-backfilled version of a file exists (OpenCode output), use it
# INSTEAD of the raw version — same underlying trajectories, don't double up.
stems = {os.path.basename(f).rsplit('.', 1)[0] for f in files}
skip = {s for s in stems if f'{s}-reasoning' in stems and not s.endswith('-reasoning')}
if skip:
    print('skipping raw file(s), reasoning-backfilled version found instead:', skip)

slices = {k: [] for k in MIX}
for f in files:
    stem = os.path.basename(f).rsplit('.', 1)[0]
    if stem in skip:
        continue
    key = SLICE_OF_FILE.get(stem)
    if key is None:
        print('skipping unknown file', f); continue
    seen_ids = set()
    for line in open(f, encoding='utf-8'):
        try: rec = json.loads(line)
        except: continue
        if rec.get('id') in seen_ids: continue   # batches append -> dedupe
        seen_ids.add(rec.get('id'))
        if rec.get('messages'): slices[key].append(rec)

avail = {k: len(v) for k, v in slices.items()}
print('available:', avail)
norm = sum(share for k, share in MIX.items() if avail.get(k))
mix = []
for k, share in MIX.items():
    want = int(TARGET_TOTAL * share / norm)
    take = random.sample(slices[k], min(want, len(slices[k]))) if slices[k] else []
    mix.extend(take)
    print(f'{k}: want {want}, took {len(take)}')
random.shuffle(mix)
print('total mix:', len(mix))

In [ ]:
# ── Model + tokenizer ──────────────────────────────────────────────────────
# Qwen3.5-9B is a vision-language model, so load it through FastVisionModel.
# We train on text ONLY, so vision layers are frozen (finetune_vision_layers=False):
# nudging the visual pathway with zero visual signal can only degrade it.
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = FINETUNE_VISION,   # False — keep inherited vision intact
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0,
    bias='none', use_gradient_checkpointing='unsloth', random_state=SEED,
)

# Sanity check: no LoRA adapter should be attached to a vision/visual module.
vision_hits = [n for n, _ in model.named_parameters()
               if 'lora' in n.lower() and any(k in n.lower() for k in ('visual', 'vision', 'image'))]
print('trainable LoRA params on vision modules:', len(vision_hits), '(expected 0)')
if vision_hits:
    print('  WARNING — vision layers are being trained; check finetune_vision_layers:', vision_hits[:3])

In [ ]:
# ── Render with Qwen3 chat template (tools included) ───────────────────────
# Our schema is already OpenAI-style messages + tools; Qwen3's template
# renders tools into the system region and tool_calls as <tool_call> JSON.
from datasets import Dataset

def render(rec):
    try:
        text = tokenizer.apply_chat_template(
            rec['messages'], tools=rec.get('tools') or None,
            tokenize=False, add_generation_prompt=False,
        )
        return {'text': text}
    except Exception as e:
        return {'text': ''}

ds = Dataset.from_list(mix).map(render, remove_columns=None)
before = len(ds)
ds = ds.filter(lambda r: len(r['text']) > 0)
# Drop samples that exceed MAX_SEQ after tokenization (truncating a trajectory
# mid-tool-call teaches garbage). Report what we lose.
def short_enough(r):
    return len(tokenizer(r['text'], add_special_tokens=False).input_ids) <= MAX_SEQ
ds = ds.filter(short_enough, num_proc=2)
print(f'render failures: {before - len(ds)} dropped (template errors + over {MAX_SEQ} tokens)')
print('final training samples:', len(ds))
print('\n--- sample render (first 1200 chars) ---\n', ds[0]['text'][:1200])

In [ ]:
# ── Train (loss on assistant tokens only) ──────────────────────────────────
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field='text',
    args=SFTConfig(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine',
        warmup_ratio=0.03, logging_steps=5, save_steps=SAVE_STEPS,
        save_total_limit=2, output_dir=OUTPUT_DIR, seed=SEED,
        max_seq_length=MAX_SEQ, packing=False,
        bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
        optim='adamw_8bit', report_to='none',
    ),
)
# Mask everything except assistant responses (incl. their <tool_call> blocks).
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
stats = trainer.train(resume_from_checkpoint=RESUME)
print(stats)

In [ ]:
# ── Save adapters (small, always) + merged 16-bit (for vLLM serving) ───────
FINAL = f'{OUTPUT_DIR}/lumen-1.3-lora'
model.save_pretrained(FINAL); tokenizer.save_pretrained(FINAL)
print('LoRA adapter saved to', FINAL)
if not TINY:
    # ~16GB write — needs Drive space; this is what the lumen endpoint serves.
    model.save_pretrained_merged(f'{OUTPUT_DIR}/lumen-1.3-merged', tokenizer, save_method='merged_16bit')
    print('merged model saved')

In [ ]:
# ── Sanity check: does it still call tools + reason? ───────────────────────
FastVisionModel.for_inference(model)
probe = [
    {'role': 'system', 'content': 'You are Lumen, an AI assistant made by Sennoric Labs.'},
    {'role': 'user', 'content': 'The test in test.js is failing. Find the bug in lib.js and fix it.'},
]
inputs = tokenizer.apply_chat_template(probe, tokenize=True,
                                       add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=400, temperature=0.3, do_sample=True)
text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False)
print(text)
print('\n<think> present:', '<think>' in text, '| tool_call present:', '<tool_call>' in text)

# Vision regression check — confirm the frozen vision tower still works after SFT.
# Feed any image and ask for a description; garbled output means vision got damaged.
print('\n--- vision spot-check: run manually with a real image before shipping ---')

## After training

- **Serve**: load `lumen-1.3-merged` in vLLM with `--enable-auto-tool-choice --tool-call-parser hermes`
  behind the existing lumen endpoint (`api.sennoric.com/v1`). Sennoric needs zero changes for text —
  `/model lumen` already points there.
- **Multimodal serving is extra work, not automatic.** The weights support images/video, but the
  serving stack doesn't yet: vLLM needs multimodal flags, `api-proxy-cf` must accept OpenAI-style
  `image_url` content parts in `/v1/chat/completions`, `chat.html` needs upload UI, and image tokens
  need billing rates. Track as its own task.
- **Eval**: run a SWE-bench-style held-out set through the deployed model and compare pass-rate
  against base Qwen3.5-9B served the same way. (Eval harness is a separate, not-yet-built piece.)
- **Regression checks before shipping**:
  - plain chat (a few ordinary questions) — no catastrophic forgetting
  - `<think>` still appears on complex prompts (thinking is ON by default in Qwen3.5)
  - **vision still works** — feed an image and ask for a description; compare against base model.
    This is the check that catches an accidental unfreezing of the vision tower.